# Explore Attribution Graph

Load a circuit-tracer attribution graph (`.pt`) and inspect its nodes, edges,
scores, and structure. Optionally launch the interactive visualisation server.

In [ ]:
# ── 0. Load the graph ────────────────────────────────────────────────────────────
import torch
from circuit_tracer import Graph

GRAPH_PATH = "../activations/circuit_graph_gemma27b_custom_prompt.pt"

graph = Graph.from_pt(GRAPH_PATH)
print(f"Input string ({graph.n_pos} tokens):")
print(graph.input_string[:300], "..." if len(graph.input_string) > 300 else "")
print(f"\nLogit targets: {graph.logit_targets}")
print(f"Logit probabilities: {graph.logit_probabilities}")
print(f"Active features: {graph.active_features.shape[0]}")
print(f"Adjacency matrix: {graph.adjacency_matrix.shape}")

In [ ]:
# ── 1. Graph quality scores ──────────────────────────────────────────────────────
from circuit_tracer.graph import compute_graph_scores

replacement_score, completeness_score = compute_graph_scores(graph)
print(f"Replacement score:  {replacement_score:.4f}  (fraction of logit influence through features vs errors)")
print(f"Completeness score: {completeness_score:.4f}  (fraction of non-error inputs weighted by influence)")

In [ ]:
# ── 2. Prune the graph & summarise ───────────────────────────────────────────────
from circuit_tracer.graph import prune_graph

NODE_THRESHOLD = 0.8
EDGE_THRESHOLD = 0.98

pruned = prune_graph(graph, node_threshold=NODE_THRESHOLD, edge_threshold=EDGE_THRESHOLD)

n_features = graph.active_features.shape[0]
n_layers = graph.cfg.n_layers
n_pos = graph.n_pos

# Node index ranges in the adjacency matrix
feat_end = n_features
error_end = feat_end + n_layers * n_pos
token_end = error_end + n_pos
# logit nodes: token_end onward

kept_feat = pruned.node_mask[:feat_end].sum().item()
kept_error = pruned.node_mask[feat_end:error_end].sum().item()
kept_token = pruned.node_mask[error_end:token_end].sum().item()
kept_logit = pruned.node_mask[token_end:].sum().item()
kept_edges = pruned.edge_mask.sum().item()

print(f"Pruned graph (node_thresh={NODE_THRESHOLD}, edge_thresh={EDGE_THRESHOLD}):")
print(f"  Feature nodes kept: {kept_feat} / {n_features}")
print(f"  Error nodes kept:   {kept_error} / {n_layers * n_pos}")
print(f"  Token nodes kept:   {kept_token} / {n_pos}")
print(f"  Logit nodes kept:   {kept_logit} / {len(graph.logit_targets)}")
print(f"  Edges kept:         {int(kept_edges)}")

In [ ]:
# ── 3. Top feature nodes by influence ────────────────────────────────────────────
import pandas as pd

# Influence scores from pruning (higher = more important for output)
scores = pruned.cumulative_scores[:n_features]
activations = graph.activation_values

# Build a dataframe of feature nodes
feat_data = graph.active_features  # (n_active, 3): layer, pos, feat_idx

rows = []
for i in range(n_features):
    layer, pos, feat_idx = feat_data[i].tolist()
    rows.append({
        "node_idx": i,
        "layer": int(layer),
        "position": int(pos),
        "feature_idx": int(feat_idx),
        "activation": float(activations[i]) if i < len(activations) else 0.0,
        "influence": float(scores[i]),
        "kept": bool(pruned.node_mask[i]),
    })

feat_df = pd.DataFrame(rows)
feat_df_sorted = feat_df.sort_values("influence", ascending=False)

print(f"Top 20 feature nodes by influence:\n")
print(feat_df_sorted.head(20).to_string(index=False))

In [ ]:
# ── 4. Top edges (feature → logit) ───────────────────────────────────────────────
# Extract the strongest direct feature-to-logit connections
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("google/gemma-3-27b-it")
input_token_strs = [tokenizer.decode(tid) for tid in graph.input_tokens.tolist()]

adj = graph.adjacency_matrix
logit_start = token_end
n_logits = len(graph.logit_targets)

# Logit rows in adjacency: rows logit_start .. logit_start+n_logits
# Columns 0..feat_end are feature sources
feat_to_logit = adj[logit_start:logit_start + n_logits, :feat_end]  # (n_logits, n_features)

edge_rows = []
for logit_i in range(n_logits):
    target = graph.logit_targets[logit_i]
    weights = feat_to_logit[logit_i]
    top_k = torch.topk(weights.abs(), k=min(10, n_features))
    for rank, (val_idx, feat_node_idx) in enumerate(zip(top_k.values, top_k.indices)):
        fi = feat_node_idx.item()
        layer, pos, feat_idx = feat_data[fi].tolist()
        edge_rows.append({
            "logit_token": target.token_str,
            "logit_prob": float(graph.logit_probabilities[logit_i]),
            "src_layer": int(layer),
            "src_pos": int(pos),
            "src_input_token": input_token_strs[int(pos)] if int(pos) < len(input_token_strs) else "?",
            "src_feature": int(feat_idx),
            "weight": float(weights[fi]),
        })

edge_df = pd.DataFrame(edge_rows)
edge_df_sorted = edge_df.reindex(edge_df["weight"].abs().sort_values(ascending=False).index)

print(f"Top 20 feature→logit edges by |weight|:\n")
print(edge_df_sorted.head(20).to_string(index=False))

In [ ]:
# ── 5. Token-level input attribution ─────────────────────────────────────────────
# Which input token embeddings contribute most to the output?

# Token embed rows in adjacency: logit rows, token embed columns
token_to_logit = adj[logit_start:logit_start + n_logits, error_end:token_end]  # (n_logits, n_pos)

# Aggregate across logit targets (weighted by logit probability)
logit_probs = graph.logit_probabilities.unsqueeze(1)  # (n_logits, 1)
token_importance = (token_to_logit.abs() * logit_probs).sum(dim=0)  # (n_pos,)

top_tokens = torch.topk(token_importance, k=min(20, n_pos))

print("Top 20 input tokens by direct logit attribution:\n")
for val, idx in zip(top_tokens.values, top_tokens.indices):
    pos = idx.item()
    tok = input_token_strs[pos] if pos < len(input_token_strs) else "?"
    print(f"  pos {pos:>4}  token={tok!r:<15}  importance={val.item():.6f}")

In [ ]:
# ── 6. Feature distribution by layer ─────────────────────────────────────────────
# How many active (kept) features come from each layer?

kept_feats = feat_df[feat_df["kept"]]
layer_counts = kept_feats.groupby("layer").size().reset_index(name="count")
layer_influence = kept_feats.groupby("layer")["influence"].sum().reset_index(name="total_influence")
layer_stats = layer_counts.merge(layer_influence, on="layer")

print("Kept features per layer:\n")
print(layer_stats.to_string(index=False))

In [ ]:
# ── 7. Generate JSON & launch interactive visualisation ────────────────────────────
# This creates the graph files and starts a local server for the circuit-tracer UI.
# Uncomment and run to launch.

# from circuit_tracer.utils.create_graph_files import create_graph_files
# from circuit_tracer.frontend.local_server import serve
#
# GRAPH_DIR = "/tmp/circuit_graphs"
# create_graph_files(
#     graph_or_path=graph,
#     slug="custom-prompt-gemma27b",
#     output_path=GRAPH_DIR,
#     node_threshold=NODE_THRESHOLD,
#     edge_threshold=EDGE_THRESHOLD,
# )
# print(f"Graph files written to {GRAPH_DIR}")
#
# server = serve(data_dir=GRAPH_DIR, port=8032)
# print("Interactive visualisation at http://localhost:8032")